In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import torch
import json
import os
import faiss
from tqdm import tqdm
from transformers import pipeline
MAX_CTX_CHARS = 1200


path_to_doc_chuncks = '/Users/tylergarfield/Desktop/undergraduate/CSC483/Rags_To_Searches/corpus/statpearls/chunk'

objc[52337]: Class AVFFrameReceiver is implemented in both /Users/tylergarfield/gemini_chats/new_mps_env/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x17ec883a8) and /Users/tylergarfield/gemini_chats/new_mps_env/lib/libavdevice.61.3.100.dylib (0x31b0147a0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[52337]: Class AVFAudioReceiver is implemented in both /Users/tylergarfield/gemini_chats/new_mps_env/lib/python3.11/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x17ec883f8) and /Users/tylergarfield/gemini_chats/new_mps_env/lib/libavdevice.61.3.100.dylib (0x31b0147f0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


In [2]:
# List out all the articles containing the chuncks to index.
docs_in_durr = os.listdir(path_to_doc_chuncks)


embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embedder.max_seq_length = 512
contents = []
titles = []

for file in tqdm(docs_in_durr,total=len(docs_in_durr)):
    
    with open(f'{path_to_doc_chuncks}/{file}','r',encoding="utf-8") as f:
            
        for line in f:
            
            curr_art_chunk = json.loads(line)

            contents.append(curr_art_chunk['content'])
            titles.append(curr_art_chunk['title'])


100%|█████████████████████████████████████████████████████████████████████████| 9632/9632 [00:04<00:00, 2168.35it/s]


In [3]:
embeddings = embedder.encode(
                contents,
                batch_size=256,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
).astype("float32")


dim = embeddings.shape[1]

Batches:   0%|          | 0/1411 [00:00<?, ?it/s]

In [4]:
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)

In [28]:
nlist = 10
quant = faiss.IndexFlatIP(dim)
index = faiss.IndexIVFFlat(quant, dim, nlist,faiss.METRIC_INNER_PRODUCT)

In [ ]:
index.train(embeddings)
index.add(embeddings)

In [7]:
def retrieve(query: str,top_k) -> list[str]:
    """Return top-k passage strings for a query."""
    q_emb = embedder.encode(
    [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    _, indices = index.search(q_emb, top_k)
    return [contents[i][:MAX_CTX_CHARS] for i in indices[0] if i >= 0]

In [20]:
ret_res = ' '.join(retrieve('Most common site of Mucosa associated lymphoid tissue (MALT) is _________',50))

In [23]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device="mps")

Device set to use mps


In [21]:
ret_res

"The human immune system is a complicated marvel that has evolved to demonstrate wide distribution through all the systems of the body (eg, the skin has the Langerhans cells, the liver's Kupffer cells, and the adenoids of the nasopharynx). One such improvisation of the immune system is the MALT, the mucosa-associated lymphoid tissue. The MALT of the gut is the gut-associated lymphoid tissue (GALT); this article is about a part of the GALT—the Peyer patches. Extranodal marginal zone lymphoma of mucosa-associated lymphoid tissue (MALT lymphoma) The significance of mucosal-associated lymphoid tissue (MALT) has recently gained prominence in understanding the pathology of various diseases, especially IgAN, as IgA production predominantly occurs in mucosal tissue. The primary role of MALT is to provide defense against environmental toxins and microbes. MALT tissue is predominantly located in the gut, particularly in Peyer's patches and lymphoid follicles, as well as in the tonsils, both of w

In [24]:
summary = summarizer(
    ret_res, 
    max_length=130,   # Don't let the summary get longer than this
    min_length=30,    # Force the summary to be at least this long
    do_sample=False   # Keep it factual, don't get "creative"
)

In [25]:
summary[0]['summary_text']

"MALT lymphoma (MALT) is a slow-growing B-cell lymphoma. It is caused by the immune system's failure to fight off infection. MALT lymphomas can be treated with chemotherapy or radiotherapy."